# L29 · 成本与性能优化：工业级 AI

**学习目标**
- 理解 AI 系统的三大成本：钱（token）、时间（延迟）、稳定（错误率）
- 掌握三种优化手段：缓存、批处理、大小模型路由
- 亲手量化「优化前后」的成本差异

**前置依赖**：L21（工具）、L27（可观测）  
**预计时长**：50 分钟  
**技术栈**：纯 Python（离线模拟成本，无需 LLM）

---

## 概念讲解：省下来的，都是利润

Demo 跑通 ≠ 能上线。真实 AI 服务每天几百万次调用，每省一分钱都是真金白银。三大杠杆：

1. **缓存**：相同问题不重复调用模型（命中缓存 = 0 成本 0 延迟）
2. **批处理**：把 100 个请求合并一次推理，摊薄开销
3. **路由**：简单问题用便宜小模型，难题才上贵的大模型

本课我们用一个「成本核算器」量化这些手段的效果。

## 第一步：模拟「调用成本模型」

In [ ]:
import time, random

BIG_COST, SMALL_COST = 0.01, 0.001   # 每次调用的钱（元）
BIG_LAT, SMALL_LAT = 0.5, 0.1        # 每次延迟（秒）

cache = {}
def call_llm(q, use_small=False, use_cache=True):
    if use_cache and q in cache:
        return cache[q], 0.0, 0.0          # 命中缓存：零成本零延迟
    cost = SMALL_COST if use_small else BIG_COST
    lat = SMALL_LAT if use_small else BIG_LAT
    time.sleep(min(lat, 0.02))            # 演示用，压缩等待
    ans = f"答:{q[:6]}"
    cache[q] = ans
    return ans, cost, lat
print("✅ 成本模型就绪：大模型¥0.01/次，小模型¥0.001/次")

## 第二步：对比「裸调」vs「加缓存+路由」

## 第三步：路由策略 —— 简单问题走小模型

In [ ]:
def router(q):
    # 简单问题（含？且短）走小模型
    return len(q) < 12

queries = ["你好", "北京天气？", "解释量子纠缠", "谢谢", "今天几号？"] * 20  # 100 次

# 方案A：全部大模型，无缓存
cost_a = lat_a = 0
for q in queries:
    _, c, l = call_llm(q, use_small=False, use_cache=False)
    cost_a += c; lat_a += l

# 方案B：缓存 + 路由
cost_b = lat_b = 0
for q in queries:
    _, c, l = call_llm(q, use_small=router(q), use_cache=True)
    cost_b += c; lat_b += l

print(f"方案A（裸调）：成本 ¥{cost_a:.2f}，总延迟 {lat_a:.1f}s")
print(f"方案B（缓存+路由）：成本 ¥{cost_b:.2f}，总延迟 {lat_b:.1f}s")
print(f"💰 节省：{(1-cost_b/cost_a)*100:.0f}% 成本，{(1-lat_b/lat_a)*100:.0f}% 延迟")

# 🎯 AHA 顿悟单元格：你的「AI 成本仪表盘」

运行下面代码。你会看到一个**成本优化对比仪表盘**：裸调 vs 优化两栏并排，
用进度条直观显示省了多少钱、多少延迟。改 `queries` 的重复度或数量，看优化收益变化。

> 真实 AI 公司靠这套思路，把每月账单从百万压到十万。你刚写的「路由+缓存」策略，就是他们的日常武器。

In [ ]:
# ===== 运行我！看成本优化仪表盘 =====
print("  📊 AI 成本优化仪表盘\n")
print("  " + "=" * 44)
print(f"  {'指标':<14}{'方案A 裸调':>14}{'方案B 优化':>14}")
print(f"  {'总成本':<12}¥{cost_a:>10.2f}  ¥{cost_b:>10.2f}")
print(f"  {'总延迟':<12}{lat_a:>11.1f}s  {lat_b:>11.1f}s")
print("  " + "=" * 44)
save_cost = (1 - cost_b/cost_a) * 100
save_lat = (1 - lat_b/lat_a) * 100
print(f"  💡 缓存命中 + 小模型路由，省下 {save_cost:.0f}% 成本、{save_lat:.0f}% 延迟")
print("  " + "█" * int(save_cost/5) + f"  {save_cost:.0f}%")
print("  ✨ 你掌握了工业级 AI 的『省钱三板斧』：缓存、批处理、路由！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：成本/延迟/质量三角权衡；路由策略的判定依据（本课用长度简化，真场景用复杂度分类）。  
**易错点**：`time.sleep` 在实际演示中压缩到 0.02s 防卡顿；缓存键为原始 query 字符串（真实用 embedding 近邻）。  
**AHA 机制**：成本对比仪表盘+进度条，强「工程省钱」实感。  
**衔接**：L30 MLOps（把优化做成常态监控）；L37 综合项目（成本约束是项目硬指标）。  
**依赖**：纯 Python 标准库。  
**SOTA 实践**：提及语义缓存（GPTCache）、模型路由（如用小模型做 80% 流量）、批推理（vLLM continuous batching）。

# 📚 作业 / 下一步

1. 把 `queries` 的重复度降低（更少重复），看缓存收益下降。
2. 加一个「批处理」函数：一次接收 10 个问题合并推理。
3. 下一课 **L30 MLOps：让 AI 系统稳定运转** —— 把模型、监控、回滚串成一条工业流水线。